<a href="https://colab.research.google.com/github/Atkiya/CSE475_ML/blob/main/Task_2/GFLOPs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install thop --quiet

import torch
import torch.nn as nn
from thop import profile, clever_format

#TinyMyo

In [10]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
            nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
        )
    def forward(self, x):
        return self.block(x)

class TinyMyo(nn.Module):
    def __init__(self, in_channels=48, num_classes=17):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(in_channels, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.head[0](x)
        x = x.squeeze(-1)
        x = self.head[1](x)
        return x


MODEL_PATH = "/content/drive/MyDrive/CSE475/tinymyo_90_10.pt"
model = TinyMyo(in_channels=48, num_classes=17)
state_dict = torch.load(MODEL_PATH, map_location="cpu")
model.load_state_dict(state_dict)
model.eval()


INPUT_SHAPE = (1, 48, 200)
dummy = torch.randn(*INPUT_SHAPE)

macs, params = profile(model, inputs=(dummy,), verbose=False)
flops = macs * 2

macs_fmt, params_fmt = clever_format([macs, params], "%.4f")
flops_fmt, _         = clever_format([flops, params], "%.4f")

print(f" TinyMyo ")
print(f"{'Parameters':<16}: {params_fmt}")
print(f"{'MACs':<16}: {macs_fmt}")
print(f"{'GFLOPs':<16}: {flops_fmt}")

 TinyMyo 
Parameters      : 102.9290K
MACs            : 20.3543M
GFLOPs          : 40.7086M


#Deep CNN

In [11]:
class DeepCNN(nn.Module):
    def __init__(self, in_channels=48, num_classes=17):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            # Block 2
            nn.Conv1d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            # Block 3
            nn.Conv1d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            # Block 4
            nn.Conv1d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(0.5), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.head(self.features(x))

deep_cnn = DeepCNN().eval()

macs, params = profile(deep_cnn, inputs=(dummy,), verbose=False)
flops = macs * 2
p, _ = clever_format([params, 0], "%.4f")
m, _ = clever_format([macs,   0], "%.4f")
f, _ = clever_format([flops,  0], "%.4f")
print(f" Deep CNN")
print(f"  Parameters : {p}")
print(f"  MACs       : {m}")
print(f"  GFLOPs     : {f}")

 Deep CNN
  Parameters : 431.7610K
  MACs       : 14.6848M
  GFLOPs     : 29.3696M


# ALR CNN

In [12]:
class _ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, pool=True):
        super().__init__()
        layers = [
            nn.Conv1d(in_ch, out_ch, kernel, padding=kernel // 2, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, kernel, padding=kernel // 2, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool1d(2))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class AttentionBlock(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(in_ch, in_ch // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv1d(in_ch // 4, in_ch, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return x * self.attention(x)

class ALR_CNN(nn.Module):
    def __init__(self, n_channels: int, n_classes: int, dropout: float = 0.3):
        super().__init__()
        self.n_channels = n_channels
        self.features = nn.Sequential(
            _ConvBlock(n_channels, 64,  kernel=7, pool=True),
            AttentionBlock(64),
            _ConvBlock(64,        128, kernel=5, pool=True),
            AttentionBlock(128),
            _ConvBlock(128,       256, kernel=3, pool=False),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(128, n_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)

# Load weights
N_CHANNELS  = 48
N_CLASSES   = 17
WIN_SAMPLES = 400  # 200 ms × 2000 Hz

MODEL_PATH = "/content/drive/MyDrive/CSE475/alr_cnn.pt"

model = ALR_CNN(n_channels=N_CHANNELS, n_classes=N_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()


dummy = torch.randn(1, N_CHANNELS, WIN_SAMPLES)
macs, params = profile(model, inputs=(dummy,), verbose=False)
flops = macs * 2

p, _ = clever_format([params, 0], "%.4f")
m, _ = clever_format([macs,   0], "%.4f")
f, _ = clever_format([flops,  0], "%.4f")

print(f" ALR-CNN ")
print(f"  Parameters : {p}")
print(f"  MACs       : {m}")
print(f"  GFLOPs     : {f}")

 ALR-CNN 
  Parameters : 515.3290K
  MACs       : 74.8488M
  GFLOPs     : 149.6977M
